# Text-to-SVG Generation: RAG-Augmented Training Pipeline

**Approach:** Retrieval-Augmented Generation (RAG) for SVG code editing 
**Idea:** For 50% of training samples, retrieve a semantically similar SVG from the training set and teach the model to *modify* it to match the target prompt. The other 50% trains standard from-scratch generation.

**Dependencies:** `sentence-transformers` for prompt embeddings, `unsloth` for LoRA fine-tuning

## 1. Configuration & Imports

In [ ]:
import torch
import random
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from transformers import TrainingArguments
from trl import SFTTrainer

CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct",
    "max_seq_length": 5120,
    "lora_r": 64,
    "lora_alpha": 128,
    "learning_rate": 1e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "output_dir": "./svg_v1",
}
SEED = 42

SYSTEM_PROMPT = (
    "You are an expert vector graphics designer. Generate highly compact, parseable, and valid SVG code. "
    "If a reference SVG is provided, intelligently modify its colors, shapes, and coordinates to match the new request. "
    "If no reference is provided, generate the SVG from scratch."
)

## 2. RAG Embedding Index

Compute sentence embeddings for all training prompts using a lightweight model (all-MiniLM-L6-v2). These embeddings are used at training time to find semantically similar SVGs, and saved for reuse during inference.

In [ ]:
print("Loading dataset and creating embeddings...")
train_df = pd.read_csv("train_complex.csv")

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
embeddings = embedder.encode(train_df['prompt'].tolist(), convert_to_tensor=True)
torch.save(embeddings, "train_embeddings.pt")
print(f"Embeddings computed and saved: {embeddings.shape}")

## 3. RAG-Augmented SFT Formatting

For each training sample, with 50% probability we retrieve the most similar (but not identical) SVG from the training set and include it as a reference in the prompt. This teaches the model two skills:
1. **Edit mode:** Modify an existing SVG to match a new description
2. **Scratch mode:** Generate from scratch when no good reference exists

In [ ]:
def get_similar_example(idx, threshold=0.85):
    """Find the most similar training example by prompt embedding cosine similarity."""
    sims = torch.nn.functional.cosine_similarity(embeddings[idx].unsqueeze(0), embeddings)
    sims[idx] = 0.0  # Don't match with itself
    
    best_idx = sims.argmax().item()
    best_score = sims[best_idx].item()
    
    if 0.5 < best_score < threshold:
        return train_df.iloc[best_idx]
    return None

def format_rag_sft_text(example, idx):
    """Format training example with optional RAG reference injection."""
    prompt = example['prompt']
    target_svg = example['svg']
    
    # 50% chance: provide a reference SVG (edit mode)
    if random.random() < 0.5:
        ref_example = get_similar_example(idx)
        if ref_example is not None:
            text = (
                f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\nReference SVG:\n{ref_example['svg']}\n\n"
                f"Modify the reference to match this new request: {prompt}<|im_end|>\n"
                f"<|im_start|>assistant\n{target_svg}<|im_end|>"
            )
            return {"text": text}
    
    # 50% fallback: generate from scratch
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{target_svg}<|im_end|>"
    )
    return {"text": text}

train_ds = Dataset.from_pandas(train_df)
train_ds = train_ds.map(format_rag_sft_text, with_indices=True, remove_columns=train_ds.column_names)
print(f"Formatted {len(train_ds)} training examples")

## 4. Model & LoRA Setup

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=False,
)

tokenizer = get_chat_template(tokenizer, chat_template="chatml")
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if len(tokenizer) > model.config.vocab_size:
    model.resize_token_embeddings(len(tokenizer))

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## 5. Training

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=TrainingArguments(
        output_dir=CONFIG["output_dir"],
        num_train_epochs=CONFIG["num_train_epochs"],
        per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        learning_rate=CONFIG["learning_rate"],
        warmup_steps=CONFIG["warmup_steps"],
        weight_decay=CONFIG["weight_decay"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        save_strategy="steps",
        save_steps=500,
        save_total_limit=5,
        optim="adamw_8bit",
        seed=SEED,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("Starting training...")
trainer.train()

model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print("Training complete. Model saved.")